In [1]:
from astropy.coordinates import SkyCoord
import astropy.units as u
import json
import drms
from astropy.time import Time
from pathlib import Path
import sunpy.map
import matplotlib.pyplot as plt
import os
import numpy as np
from astropy.io import fits
from PIL import Image
import pandas as pd
from astropy.time import Time

In [2]:
df=pd.read_excel('../dataset/M级耀斑.xlsx')
#删掉没用的几行
#df.drop(df.index[8:16], inplace=True)
#转化成datetime格式
df['Start Time'] = pd.to_datetime(df['Start Time'])

In [3]:
#检查发现TAI和UTC两个时间其实是不一样的
from astropy.time import Time
#TAI是原子钟时间，比UTC要多35秒的闰秒差距
def utc_to_tai_time(t_utc):
    return Time(t_utc, scale="utc").tai

t1=utc_to_tai_time(df['Start Time'][0:3])
print(df['Start Time'][0:3])
print(t1)

0   2012-07-01 19:11:00
1   2012-07-02 00:26:00
2   2012-07-02 10:43:00
Name: Start Time, dtype: datetime64[ns]
['2012-07-01T19:11:35.000000000' '2012-07-02T00:26:35.000000000'
 '2012-07-02T10:43:35.000000000']


__<font color=cyan>下面这一段是下载AIA 131,211,304和白光的数据</font>__

In [4]:
import drms
from pathlib import Path

JSOC_EMAIL = "652024260007@smail.nju.edu.cn"
HMI_SERIES = "hmi.Ic_45s"
OUT_DIR = Path("HMI")
OUT_DIR.mkdir(exist_ok=True)
client = drms.Client()

In [ ]:
from pathlib import Path
from astropy.time import Time
import astropy.units as u
import drms

client = drms.Client(email=JSOC_EMAIL)

ROOT_DIR = Path(r"C:\Learning\PHD2nd\sunspotscar\data\M")

AIA_SERIES = "aia.lev1_euv_12s"
AIA_WAVELENGTHS = [131, 211, 304]
AIA_CADENCE = "36s"

HMI_SERIES_FULL = "hmi.Ic_45s"
HMI_SEGMENT = "continuum"
HMI_FOLDER_NAME = "hmi.Ic_45s"


def start_time_to_folder_name(t_utc):
    """
    用 Start Time 的 UTC 时间生成事件文件夹名。
    """
    t = Time(t_utc, scale="utc")
    return t.utc.strftime("%Y%m%d_%H%M%S_UTC")


def get_40min_tai_window(t_utc):
    """
    Start Time 前30分钟到后10分钟，返回 JSOC TAI 时间字符串。
    """
    t_utc = Time(t_utc, scale="utc")
    t_tai = t_utc.tai

    t_start = t_tai - 30 * u.minute
    t_end = t_tai + 10 * u.minute

    t_start_str = t_start.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    t_end_str = t_end.strftime("%Y.%m.%d_%H:%M:%S_TAI")

    return t_utc, t_start_str, t_end_str


def export_and_download(qstr, out_dir):
    """
    执行 JSOC export 并下载到指定目录。
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"Exporting: {qstr}")
    print(f"Save to: {out_dir}")

    export_req = client.export(
        qstr,
        method="url_quick",
        protocol="fits",
        email=JSOC_EMAIL,
    )

    if len(export_req.urls) == 0:
        print("  [WARNING] export returned no URLs")
        return []

    dl = export_req.download(str(out_dir))
    files = [Path(f) for f in dl.download]

    print(f"  Saved {len(files)} FITS files")
    return files



def download_event_all_channels(t_utc, root_dir=ROOT_DIR):
    t_utc, t_start_str, t_end_str = get_40min_tai_window(t_utc)

    event_name = start_time_to_folder_name(t_utc)
    event_dir = root_dir / event_name

    print("\n" + "=" * 80)
    print(f"Start Time UTC: {t_utc.isot}")
    print(f"TAI window: {t_start_str} - {t_end_str}")
    print(f"Event dir: {event_dir}")

    def export_and_download(qstr, out_dir):
        out_dir.mkdir(parents=True, exist_ok=True)

        existing_files = list(out_dir.glob("*.fits"))
        if len(existing_files) > 0:
            print(f"  [SKIP] {out_dir} already has {len(existing_files)} FITS files")
            return existing_files

        print(f"Exporting: {qstr}")
        print(f"Save to: {out_dir}")

        export_req = client.export(
            qstr,
            method="url_quick",
            protocol="fits",
            email=JSOC_EMAIL,
        )

        if len(export_req.urls) == 0:
            print("  [WARNING] export returned no URLs")
            return []

        dl = export_req.download(str(out_dir))
        files = [Path(f) for f in dl.download]

        print(f"  Saved {len(files)} FITS files")
        return files

    result = {}

    # AIA 131 / 211 / 304: 12s -> 36s, 每3帧取1帧
    for wavelength in AIA_WAVELENGTHS:
        out_dir = event_dir / str(wavelength)

        qstr = (
            f"{AIA_SERIES}"
            f"[{t_start_str}-{t_end_str}@{AIA_CADENCE}]"
            f"[{wavelength}]"
            f"{{image}}"
        )

        try:
            result[str(wavelength)] = export_and_download(qstr, out_dir)
        except Exception as e:
            print(f"  [ERROR] AIA {wavelength}: {e}")
            result[str(wavelength)] = None

    # HMI Ic 45s continuum: 保持原始45s cadence
    hmi_out_dir = event_dir / HMI_FOLDER_NAME

    hmi_qstr = (
        f"{HMI_SERIES_FULL}"
        f"[{t_start_str}-{t_end_str}]"
        f"{{{HMI_SEGMENT}}}"
    )

    try:
        result[HMI_FOLDER_NAME] = export_and_download(hmi_qstr, hmi_out_dir)
    except Exception as e:
        print(f"  [ERROR] HMI: {e}")
        result[HMI_FOLDER_NAME] = None

    return result

# 批量下载
download_error_list = []
download_result = {}

for i, t in enumerate(df["Start Time"]):
    try:
        download_result[i] = download_event_all_channels(t)
    except Exception as e:
        print(f"[ERROR] index={i}, Start Time={t}, error={e}")
        download_error_list.append(i)

print("download_error_list:", download_error_list)


Start Time UTC: 2012-07-12T15:37:00.000
TAI window: 2012.07.12_15:07:35_TAI - 2012.07.12_15:47:35_TAI
Event dir: C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC
  [SKIP] C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC\131 already has 67 FITS files
  [SKIP] C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC\211 already has 67 FITS files
  [SKIP] C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC\304 already has 67 FITS files
  [SKIP] C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC\hmi.Ic_45s already has 54 FITS files

Start Time UTC: 2013-11-08T04:20:00.000
TAI window: 2013.11.08_03:50:35_TAI - 2013.11.08_04:30:35_TAI
Event dir: C:\Learning\PHD2nd\sunspotscar\data\X\20131108_042000_UTC
  [SKIP] C:\Learning\PHD2nd\sunspotscar\data\X\20131108_042000_UTC\131 already has 67 FITS files
  [SKIP] C:\Learning\PHD2nd\sunspotscar\data\X\20131108_042000_UTC\211 already has 67 FITS files
  [SKIP] C:\Learning\PHD2nd\sunspotscar\data\X\20131108_042000_UTC\

__<font color=cyan>下面这一段是下载爆发前最邻近时刻的B720s矢量磁图</font>__

In [ ]:
from pathlib import Path
from astropy.time import Time
import astropy.units as u

HMI_VECTOR_SERIES = "hmi.B_720s"
HMI_VECTOR_FOLDER_NAME = "hmi.B_720s"
ROOT_DIR = Path(r"C:\Learning\PHD2nd\sunspotscar\data\M")
JSOC_EMAIL = "19374325@buaa.edu.cn"

# 720s 矢量磁场常用核心 segments
HMI_VECTOR_SEGMENTS = ["field", "inclination", "azimuth", "disambig"]


def jsoc_trec_to_astropy_time(trec_str):
    """
    JSOC T_REC -> astropy Time(TAI)
    '2013.01.07_08:44:00_TAI' -> Time(..., scale='tai')
    """
    s = trec_str
    if isinstance(s, (bytes, bytearray)):
        s = s.decode("utf-8")
    s = str(s)

    if s.endswith("_TAI"):
        s = s[:-4]

    s = s.replace(".", "-", 2).replace("_", "T", 1)
    return Time(s, format="isot", scale="tai")


def find_previous_hmi_vector_720s_record(t_utc, lookback=2*u.hour):
    """
    找到 Start Time 之前最近的一帧 hmi.B_720s 记录。
    只返回 T_REC <= Start Time 的最近记录。
    """
    t_utc = Time(t_utc, scale="utc")
    t_tai = t_utc.tai

    t_start = (t_tai - lookback).strftime("%Y.%m.%d_%H:%M:%S_TAI")
    t_end = t_tai.strftime("%Y.%m.%d_%H:%M:%S_TAI")

    q = f"{HMI_VECTOR_SERIES}[{t_start}-{t_end}]"
    print(f"Query previous HMI vector: {q}")

    res = client.query(q, key="T_REC")

    if res is None or len(res) == 0:
        print(f"  [WARNING] No {HMI_VECTOR_SERIES} record before {t_utc.isot}")
        return None

    trec_strs = [
        s.decode("utf-8") if isinstance(s, (bytes, bytearray)) else str(s)
        for s in res["T_REC"]
    ]

    trec_times = Time([jsoc_trec_to_astropy_time(s) for s in trec_strs])

    # 只允许 <= Start Time
    valid = trec_times <= t_tai
    if not valid.any():
        print(f"  [WARNING] No previous record <= {t_utc.isot}")
        return None

    valid_indices = [i for i, ok in enumerate(valid) if ok]
    dt = t_tai - trec_times[valid]
    idx_in_valid = dt.argmin()
    idx = valid_indices[idx_in_valid]

    nearest_previous_trec = trec_strs[idx]
    print(f"  Previous nearest T_REC: {nearest_previous_trec}")
    print(f"  Delta before Start Time: {(t_tai - trec_times[idx]).to(u.minute):.2f}")

    return nearest_previous_trec


def download_hmi_vector_720s_previous(t_utc, root_dir=ROOT_DIR):
    """
    下载 Start Time 之前最近一帧 720s full-disk HMI 矢量磁图。

    存储结构：
    X / 事件日期 / hmi.B_720s / fits文件
    """
    t_utc = Time(t_utc, scale="utc")
    event_name = start_time_to_folder_name(t_utc)
    out_dir = root_dir / event_name / HMI_VECTOR_FOLDER_NAME
    out_dir.mkdir(parents=True, exist_ok=True)

    # 如果已经有文件，就跳过，避免重复下载
    existing_files = list(out_dir.glob("*.fits"))
    if len(existing_files) > 0:
        print(f"  [SKIP] {out_dir} already has {len(existing_files)} FITS files")
        return existing_files, None

    previous_trec = find_previous_hmi_vector_720s_record(t_utc)

    if previous_trec is None:
        return [], None

    segment_str = ",".join(HMI_VECTOR_SEGMENTS)

    qstr = (
        f"{HMI_VECTOR_SERIES}"
        f"[{previous_trec}]"
        f"{{{segment_str}}}"
    )

    print(f"Exporting: {qstr}")
    print(f"Save to: {out_dir}")

    export_req = client.export(
        qstr,
        method="url_quick",
        protocol="fits",
        email=JSOC_EMAIL,
    )

    if len(export_req.urls) == 0:
        print("  [WARNING] export returned no URLs")
        return [], previous_trec

    dl = export_req.download(str(out_dir))
    files = [Path(f) for f in dl.download]

    print(f"  Saved {len(files)} HMI vector FITS files")
    return files, previous_trec

In [10]:
hmi_vector_error_list = []
hmi_vector_empty_list = []
hmi_vector_trec_list = [None] * len(df)

for i, t in enumerate(df["Start Time"]):
    try:
        print("\n" + "=" * 80)
        print(f"index={i}, Start Time={t}")

        files, trec = download_hmi_vector_720s_previous(t)
        hmi_vector_trec_list[i] = trec

        if len(files) == 0:
            hmi_vector_empty_list.append(i)

    except Exception as e:
        print(f"[ERROR] index={i}, Start Time={t}, error={e}")
        hmi_vector_error_list.append(i)

print("hmi_vector_error_list:", hmi_vector_error_list)
print("hmi_vector_empty_list:", hmi_vector_empty_list)


index=0, Start Time=2012-07-12 15:37:00
Query previous HMI vector: hmi.B_720s[2012.07.12_13:37:35_TAI-2012.07.12_15:37:35_TAI]
  Previous nearest T_REC: 2012.07.12_15:36:00_TAI
  Delta before Start Time: 1.58 min
Exporting: hmi.B_720s[2012.07.12_15:36:00_TAI]{field,inclination,azimuth,disambig}
Save to: C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC\hmi.B_720s


2026-05-28 10:23:38 - drms - INFO: Export request pending. [id=JSOC_20260528_000875, status=2]
2026-05-28 10:23:38 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:23:44 - drms - INFO: Export request pending. [id=JSOC_20260528_000875, status=1]
2026-05-28 10:23:44 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:23:50 - drms - INFO: Export request pending. [id=JSOC_20260528_000875, status=1]
2026-05-28 10:23:50 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:23:56 - drms - INFO: Export request pending. [id=JSOC_20260528_000875, status=1]
2026-05-28 10:23:56 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:24:03 - drms - INFO: Export request pending. [id=JSOC_20260528_000875, status=1]
2026-05-28 10:24:03 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:24:09 - drms - INFO: Export request pending. [id=JSOC_20260528_000875, status=1]
2026-05-28 10:24:09 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:24:14 - drms - INFO: Export request pending. [id=JSOC_20

  Saved 4 HMI vector FITS files

index=1, Start Time=2013-11-08 04:20:00
Query previous HMI vector: hmi.B_720s[2013.11.08_02:20:35_TAI-2013.11.08_04:20:35_TAI]
  Previous nearest T_REC: 2013.11.08_04:12:00_TAI
  Delta before Start Time: 8.58 min
Exporting: hmi.B_720s[2013.11.08_04:12:00_TAI]{field,inclination,azimuth,disambig}
Save to: C:\Learning\PHD2nd\sunspotscar\data\X\20131108_042000_UTC\hmi.B_720s


2026-05-28 10:25:45 - drms - INFO: Export request pending. [id=JSOC_20260528_000889, status=2]
2026-05-28 10:25:45 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:25:51 - drms - INFO: Export request pending. [id=JSOC_20260528_000889, status=1]
2026-05-28 10:25:51 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:25:57 - drms - INFO: Export request pending. [id=JSOC_20260528_000889, status=1]
2026-05-28 10:25:57 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:26:03 - drms - INFO: Export request pending. [id=JSOC_20260528_000889, status=1]
2026-05-28 10:26:03 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:26:09 - drms - INFO: Export request pending. [id=JSOC_20260528_000889, status=1]
2026-05-28 10:26:09 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:26:15 - drms - INFO: Export request pending. [id=JSOC_20260528_000889, status=1]
2026-05-28 10:26:15 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:26:21 - drms - INFO: Export request pending. [id=JSOC_20

  Saved 4 HMI vector FITS files

index=2, Start Time=2013-11-10 05:08:00
Query previous HMI vector: hmi.B_720s[2013.11.10_03:08:35_TAI-2013.11.10_05:08:35_TAI]
  Previous nearest T_REC: 2013.11.10_05:00:00_TAI
  Delta before Start Time: 8.58 min
Exporting: hmi.B_720s[2013.11.10_05:00:00_TAI]{field,inclination,azimuth,disambig}
Save to: C:\Learning\PHD2nd\sunspotscar\data\X\20131110_050800_UTC\hmi.B_720s


...

  Saved 4 HMI vector FITS files

index=6, Start Time=2014-10-24 21:07:00
Query previous HMI vector: hmi.B_720s[2014.10.24_19:07:35_TAI-2014.10.24_21:07:35_TAI]
  Previous nearest T_REC: 2014.10.24_21:00:00_TAI
  Delta before Start Time: 7.58 min
Exporting: hmi.B_720s[2014.10.24_21:00:00_TAI]{field,inclination,azimuth,disambig}
Save to: C:\Learning\PHD2nd\sunspotscar\data\X\20141024_210700_UTC\hmi.B_720s


2026-05-28 10:35:10 - drms - INFO: Export request pending. [id=JSOC_20260528_000950, status=2]
2026-05-28 10:35:10 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:35:16 - drms - INFO: Export request pending. [id=JSOC_20260528_000950, status=1]
2026-05-28 10:35:16 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:35:22 - drms - INFO: Export request pending. [id=JSOC_20260528_000950, status=1]
2026-05-28 10:35:22 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:35:28 - drms - INFO: Export request pending. [id=JSOC_20260528_000950, status=1]
2026-05-28 10:35:28 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:35:33 - drms - INFO: Export request pending. [id=JSOC_20260528_000950, status=1]
2026-05-28 10:35:33 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:35:39 - drms - INFO: Export request pending. [id=JSOC_20260528_000950, status=1]
2026-05-28 10:35:39 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:35:45 - drms - INFO: Export request pending. [id=JSOC_20

  Saved 4 HMI vector FITS files

index=7, Start Time=2015-03-11 16:11:00
Query previous HMI vector: hmi.B_720s[2015.03.11_14:11:35_TAI-2015.03.11_16:11:35_TAI]
  Previous nearest T_REC: 2015.03.11_16:00:00_TAI
  Delta before Start Time: 11.58 min
Exporting: hmi.B_720s[2015.03.11_16:00:00_TAI]{field,inclination,azimuth,disambig}
Save to: C:\Learning\PHD2nd\sunspotscar\data\X\20150311_161100_UTC\hmi.B_720s


2026-05-28 10:37:16 - drms - INFO: Export request pending. [id=JSOC_20260528_000963, status=2]
2026-05-28 10:37:16 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:37:22 - drms - INFO: Export request pending. [id=JSOC_20260528_000963, status=1]
2026-05-28 10:37:22 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:37:28 - drms - INFO: Export request pending. [id=JSOC_20260528_000963, status=1]
2026-05-28 10:37:28 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:37:34 - drms - INFO: Export request pending. [id=JSOC_20260528_000963, status=1]
2026-05-28 10:37:34 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:37:40 - drms - INFO: Export request pending. [id=JSOC_20260528_000963, status=1]
2026-05-28 10:37:40 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:37:45 - drms - INFO: Export request pending. [id=JSOC_20260528_000963, status=1]
2026-05-28 10:37:45 - drms - INFO: Waiting for 5 seconds...
2026-05-28 10:37:51 - drms - INFO: Export request pending. [id=JSOC_20

  Saved 4 HMI vector FITS files
hmi_vector_error_list: []
hmi_vector_empty_list: []
